In [1]:
import numpy as np
import matplotlib.pylab as plt

#Ejercicio 1:

In [2]:

# Use esta funcion que recibe un valor x y retorna un valor f(x) donde f es la forma funcional que debe seguir su distribucion.
def mifun(x):
  x_0 = 3.0
  a = 0.01
  return np.exp(-(x**2))/((x-x_0)**2 + a**2)

# 1) Dentro de una funcion que reciba como parametros el numero de pasos y el sigma de la distribucion gausiana que va a usar
# para calcular el paso de su caminata, implemente el algortimo de Metropolis-Hastings.

# 2) Haga un histograma de los valores de x obtenidos y grafique en la misma grafica, la funcion de distribucion de probabilidad fx (Ojo, aca debe normalizar). #
#

def metropolis_hastings(sigma, pasos, f):
    x_r = np.random.uniform(-4, 4)
    datos = np.empty(pasos)
    beta = 0.0

    for i in range(pasos):
        x_nr = np.random.normal(loc=x_r, scale=sigma)
        alpha = f(x_nr)/f(x_r)
        if  alpha >= 1:
            x_r = x_nr
        else:
            beta = np.random.uniform(0, 1)
        if alpha > beta:
            x_r = x_nr
        datos[i] = x_r

    x = np.linspace(-4, 4, 1000)
    y = f(x)
    y = y / np.trapezoid(y, x)
    plt.hist(datos, bins=100, density=True)
    plt.plot(x, y)
    plt.savefig("histograma_" + str(sigma) + "_" + str(pasos) + ".pdf")
    plt.close()
    return datos

# 3) Cuando haya verificado que su codigo funciona, use los siguientes parametros:
# sigma = 5, pasos =100000
# sigma = 0.2, pasos =100000
# sigma = 0.01, pasos =100000
# sigma = 0.1, pasos =1000
# sigma = 0.1, pasos =100000
# OPCIONAL: este puede ser muy demorado dependiendo del computador: sigma = 0.1, pasos =500000

ej1 = metropolis_hastings(5, 100000, mifun)
ej2 = metropolis_hastings(0.2, 100000, mifun)
ej3 = metropolis_hastings(0.01, 100000, mifun)
ej4 = metropolis_hastings(0.1, 1000, mifun)
ej5 = metropolis_hastings(0.1, 100000, mifun)
ej6 = metropolis_hastings(0.1, 500000, mifun)

# Al ejecutar el codigo, este debe generar 6 (o 5) graficas, una para cada vez que se llama a la funcion con los parametros antes mencionados.

#Ejercicio 2:

In [10]:
# 1) lea los datos de resorte.dat y almacenelos.

resorte_data = np.genfromtxt("resorte.dat")

t = resorte_data[:, 0]
x = resorte_data[:, 1]

# Los datos corresponden a las posiciones en x de un oscilador (masa resorte) en funcion del tiempo.
# La ecuacion de movimiento esta dada por:
# x(t)=a*np.exp(-gamma*t)*np.cos(omega*t)
# Donde a, gamma, y omega son parametros.

# 2) Implemente un algoritmo que le permita, por medio de estimacion bayesiana de parametros,
# encontrar los MEJORES parametros (a, gamma y omega) tales que la ecuacion ajuste los datos. Para esto debe:

# 2a.) definir una funcion que reciba los parametros que se busca estimar y los datos de tiempo y retorne el modelo

def modelo_resorte(a, gamma, omega, t):
    return a*np.exp(-gamma*t)*np.cos(omega*t)

# 2b.) Definir una funcion que retorne la funcion de verosimilitud

def verosimilitud(x_obs, x_mod):
    chi_squared = (1.0/2.0)*np.sum((x_obs-x_mod)**2)
    return np.exp(-chi_squared)

# 2c.) Caminata

def metropolis_hastings(a_init, gamma_init, omega_init, pasos):
    datos_a = np.empty(pasos)
    datos_gamma = np.empty(pasos)
    datos_omega = np.empty(pasos)
    L = np.empty(pasos)
    beta = 0.0

    x_mod_init = modelo_resorte(a_init, gamma_init, omega_init, t)

    for i in range(pasos):
        a_newr = np.random.normal(loc=a_init, scale=0.2)
        gamma_newr = np.random.normal(loc=gamma_init, scale=0.02)
        omega_newr = np.random.normal(loc=omega_init, scale=0.4)

        x_mod_newr = modelo_resorte(a_newr, gamma_newr, omega_newr, t)

        L[i] = verosimilitud(x, x_mod_init)

        alpha = verosimilitud(x, x_mod_newr)/verosimilitud(x, x_mod_init)
        if  alpha >= 1:
            x_mod_init = x_mod_newr
            a_init = a_newr
            gamma_init = gamma_newr
            omega_init = omega_newr
        else:
            beta = np.random.uniform(0, 1)
            if alpha > beta:
                x_mod_init = x_mod_newr
                a_init = a_newr
                gamma_init = gamma_newr
                omega_init = omega_newr
        datos_a[i] = a_init
        datos_gamma[i] = gamma_init
        datos_omega[i] = omega_init

    indice_mejor_modelo = np.argmax(L)

    return datos_a, datos_gamma, datos_omega, indice_mejor_modelo

#condiciones iniciales
aini=7.12
gammaini=0.69
omegaini=18.32

#numero de pasos
iteraciones=100000

a, gamma, omega, i_max = metropolis_hastings(aini, gammaini, omegaini, iteraciones)

# 2d.) Seleccione los mejores parametros E IMPRIMA UN MENSAJE QUE DIGA: "LOS MEJORES PARAMETROS SON a=... gamma=... Y omgega=..."

print(f"LOS MEJORES PARAMETROS SON a = {a[i_max]} , gamma = {gamma[i_max]} , Y omega = {omega[i_max]}")

# 2f.) Grafique sus datos originales y su modelo con los mejores parametros. Guarde su grafica sin mostrarla en Resorte.pdf

plt.scatter(t, x, color='blue')
plt.plot(t, modelo_resorte(a[i_max], gamma[i_max], omega[i_max], t), color='red')
plt.savefig("Resorte.pdf")
plt.close()

# 3) SABIENDO QUE omega=np.sqrt(k/m), IMPRIMA UN MENSAJE DONDE EXPLIQUE SI PUEDE O NO DETERMINAR k Y m DE MANERA INDIVIDUAL USANDO EL METODO ANTERIOR. JUSTIFIQUE BIEN SU RESPUESTA (PUEDE ADEMAS HACER PRUEBAS CON EL CODIGO PARA RESPONDER ESTA PREGUNTA).

print("\n\nNo es posible determinar k y m de manera individual. El modelo depende únicamente del parámetro compuesto omega = sqrt(k/m), y existen infinitos pares de valores (k, m) que producen el mismo valor de omega y, por ende, exactamente la misma curva de ajuste x(t) y el mismo valor de verosimilitud. Para conocer k y m por separado, se necesitaria una medición experimental adicional (como medir directamente la masa).")


LOS MEJORES PARAMETROS SON a = 10.169865619056788 , gamma = 0.7310876553632758 , Y omega = 17.030092339934814


No es posible determinar k y m de manera individual. El modelo depende únicamente del parámetro compuesto omega = sqrt(k/m), y existen infinitos pares de valores (k, m) que producen el mismo valor de omega y, por ende, exactamente la misma curva de ajuste x(t) y el mismo valor de verosimilitud. Para conocer k y m por separado, se necesitaria una medición experimental adicional (como medir directamente la masa).
